The purpose of this program is to convert the filtered data to a sparse matrix, representing a graph connecting listeners with songs.

Import some essential libraries

In [1]:
import os
import csv
import numpy as np
from scipy import sparse
import sklearn

In [2]:
current_dir = os.getcwd()
project_directory, notebooks_subdirectory = os.path.split(current_dir)
if notebooks_subdirectory == "notebooks":
    os.chdir(project_directory)
    print(f"Changed working directory to project root: {project_directory}")
else:
    project_directory = current_dir
# Confirm the new current directory
os.getcwd()

Changed working directory to project root: c:\Users\dave4\vscode-projects\spotify-playlists


'c:\\Users\\dave4\\vscode-projects\\spotify-playlists'

In [3]:
data_directory = os.path.join(project_directory, "data")
if project_directory not in os.sys.path:
    os.sys.path.append(project_directory)

In [4]:
def decoded_line_generator(file):
    for line in file:
        line_str = line.decode('utf-8')
        yield line_str

In [5]:
popular_listens_path = os.path.join(data_directory, 'popular_listens.csv')

# Make a preliminary pass to get the max user and song IDs
max_user_id = 0
max_song_id = 0
total_entries = 0
with open(popular_listens_path, 'rb') as file:
    reader = csv.reader(decoded_line_generator(file))
    for row in reader:
        user_id = int(row[0])
        song_id = int(row[1])
        if user_id > max_user_id:
            max_user_id = user_id
        if song_id > max_song_id:
            max_song_id = song_id
        total_entries += 1

m = max_user_id + max_song_id + 2

# Read the data and build the sparse matrix
a = np.zeros((total_entries * 2,2), dtype=np.int32)
n = 0

with open(popular_listens_path, 'rb') as file:
    reader = csv.reader(decoded_line_generator(file))
    for row in reader:
        a[n, 0] = a[n+1, 1] = int(row[0])
        a[n, 1] = a[n+1, 0] = int(row[1]) + max_user_id + 1
        n += 2

coo_matrix = sparse.coo_matrix((np.ones(total_entries*2, dtype=np.int8), (a[:, 0], a[:, 1])), shape=(m, m))
csr_matrix = coo_matrix.tocsr()
csr_matrix

<Compressed Sparse Row sparse matrix of dtype 'int8'
	with 2703014 stored elements and shape (19600, 19600)>

In [6]:
embedding = sklearn.manifold.SpectralEmbedding(n_components=10, random_state=42, affinity='precomputed', n_jobs=-1)
transform = embedding.fit_transform(csr_matrix)

c:\Users\dave4\vscode-projects\spotify-playlists\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


In [7]:
import plotly.express as px

fig = px.scatter(x=transform[:,8], y=transform[:,9])
fig.show()